# Pipeline 4: Safehouse Effectiveness Analysis

## New Dawn Lighthouse Foundation - Explanatory Regression Model

**Business Problem:** The Lighthouse Foundation operates multiple safehouses across the Philippines, each with different capacities, staffing levels, partner organizations, and operational characteristics. Leadership needs to understand which operational factors drive better educational outcomes for residents, so they can allocate resources effectively and replicate best practices across all facilities.

**Target Variable:** `avg_education_progress` - the average education progress score across residents at each safehouse per month (continuous).

**Approach:** This is primarily an **explanatory model** because the goal is to identify *which operational factors* contribute to better education outcomes, not to predict future scores. Understanding the causal mechanisms enables evidence-based resource allocation.

We use:
- **Ch. 6 (Univariate Profiling):** Profile all safehouse-month metrics.
- **Ch. 7 (Data Preparation):** Handle skewness, missing values, and engineer capacity utilization.
- **Ch. 8 (Bivariate Analysis):** Examine each operational factor's relationship with education progress.
- **Ch. 9 (Explanatory Modeling):** Build statsmodels OLS with standardized coefficients and full diagnostics.
- **Ch. 10 (Predictive Comparison):** Compare sklearn Ridge and Lasso for robustness.
- **Ch. 15 (Deployment):** Produce comparative dashboard recommendations.

---
## Section 1: Data Loading & Initial Exploration (Ch. 5)

This pipeline joins three tables:
- **safehouse_monthly_metrics:** Monthly operational statistics per safehouse (active residents, service counts, education/health averages).
- **safehouses:** Static safehouse attributes (region, capacity, open date).
- **partner_assignments:** Which partner organizations support each safehouse and in what program areas.

The unit of analysis is **safehouse-month** -- one row per safehouse per month. This gives us repeated measures over time, allowing us to examine both between-safehouse and within-safehouse variation in education outcomes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

import sys
sys.path.insert(0, '.')
from functions import unistats, bin_categories, skew_correct, missing_fill, clean_outlier, n2n_analysis, n2c_analysis, c2c_analysis, calculate_vif

print('Libraries loaded successfully.')

In [ ]:
# Load data
metrics = pd.read_csv('../lighthouse_csv_v7/safehouse_monthly_metrics.csv', 
                       parse_dates=['month_start', 'month_end'])
safehouses = pd.read_csv('../lighthouse_csv_v7/safehouses.csv')
partners = pd.read_csv('../lighthouse_csv_v7/partner_assignments.csv')

print(f'Monthly Metrics: {metrics.shape}')
print(f'Safehouses:      {safehouses.shape}')
print(f'Partners:        {partners.shape}')

In [ ]:
metrics.head(3)

In [ ]:
safehouses.head()

In [ ]:
partners.head(3)

### 1.1 Feature Engineering

We engineer the following features by merging the three tables:

- **active_residents:** Number of residents in the safehouse that month.
- **process_recording_count:** Number of counseling sessions conducted.
- **home_visitation_count:** Number of home visits conducted.
- **incident_count:** Number of incidents reported.
- **partner_count:** Number of active partner organizations assigned to the safehouse.
- **region:** Geographic region of the safehouse (categorical).
- **capacity_utilization:** active_residents / capacity_girls -- measures how full the safehouse is. Higher utilization may strain resources.

In [ ]:
# Count active partners per safehouse
partner_counts = partners[partners['status'] == 'Active'].groupby('safehouse_id').agg(
    partner_count=('assignment_id', 'count'),
    program_areas=('program_area', 'nunique')
).reset_index()

print(f'Partner counts per safehouse:')
partner_counts

In [ ]:
# Merge metrics with safehouse attributes
df = metrics.merge(safehouses[['safehouse_id', 'region', 'capacity_girls', 'name']], 
                   on='safehouse_id', how='left')

# Merge partner counts
df = df.merge(partner_counts, on='safehouse_id', how='left')
df['partner_count'] = df['partner_count'].fillna(0)
df['program_areas'] = df['program_areas'].fillna(0)

# Calculate capacity utilization
df['capacity_utilization'] = df['active_residents'] / df['capacity_girls'].replace(0, np.nan)

print(f'Merged dataset shape: {df.shape}')
print(f'\nTarget variable - avg_education_progress:')
print(df['avg_education_progress'].describe())

In [ ]:
df.info()

In [ ]:
# Overview of safehouses in the data
safehouse_summary = df.groupby('name').agg(
    months=('metric_id', 'count'),
    avg_residents=('active_residents', 'mean'),
    avg_edu_progress=('avg_education_progress', 'mean'),
    region=('region', 'first'),
    capacity=('capacity_girls', 'first')
).sort_values('avg_edu_progress', ascending=False)

print('Safehouse Summary:')
safehouse_summary

---
## Section 2: Univariate Profiling (Ch. 6)

Chapter 6's profiling helps us understand the distribution of each operational metric across safehouse-months. Key questions:
- Is the target variable (avg_education_progress) approximately normally distributed, or does it need transformation?
- Are there operational metrics with high skewness or outliers that could distort regression results?
- Are any features nearly constant (zero variance), which would not contribute to the model?

In [ ]:
# Full univariate profile
profile = unistats(df)
profile

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(df['avg_education_progress'], bins=30, color='#A2C9E1', edgecolor='white')
axes[0].set_title(f'avg_education_progress\nskew={df["avg_education_progress"].skew():.2f}')
axes[0].set_xlabel('Average Education Progress')

axes[1].boxplot(df['avg_education_progress'].dropna())
axes[1].set_title('Box Plot')

stats.probplot(df['avg_education_progress'].dropna(), plot=axes[2])
axes[2].set_title('Q-Q Plot')

plt.suptitle('Target Variable: avg_education_progress (Ch. 6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions
feature_cols = ['active_residents', 'process_recording_count', 'home_visitation_count',
                'incident_count', 'partner_count', 'capacity_utilization', 'avg_health_score']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(feature_cols):
    if col in df.columns:
        ax = axes[i // 4, i % 4]
        df[col].hist(bins=25, ax=ax, color='#91B191', edgecolor='white')
        ax.set_title(f'{col}\nskew={df[col].skew():.2f}', fontsize=9)

# Remove empty last subplot if needed
if len(feature_cols) < 8:
    axes[1, 3].set_visible(False)

plt.suptitle('Feature Distributions (Ch. 6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Education progress over time by safehouse
fig, ax = plt.subplots(figsize=(14, 6))
for sh_name, group in df.groupby('name'):
    group_sorted = group.sort_values('month_start')
    ax.plot(group_sorted['month_start'], group_sorted['avg_education_progress'], 
            marker='o', markersize=3, label=sh_name, alpha=0.7)

ax.set_xlabel('Month')
ax.set_ylabel('Avg Education Progress')
ax.set_title('Education Progress Over Time by Safehouse')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

**Univariate Profile Interpretation (Ch. 6):**

The profiling reveals the operational landscape across the Foundation's safehouses:
- **avg_education_progress** (target): We need to check if the distribution is approximately normal. Severe skewness may warrant transformation.
- **active_residents**: Variation in safehouse occupancy reflects different facility sizes and intake patterns.
- **process_recording_count & home_visitation_count**: These service delivery metrics may be right-skewed, with some months having significantly more activity than others.
- **incident_count**: Likely right-skewed (many months with few incidents, occasional months with more).
- **capacity_utilization**: Should range between 0 and 1+. Values above 1 indicate overcrowding, which may negatively impact education outcomes.

The time-series plot reveals whether education progress is improving over time across safehouses, and whether there is substantial between-safehouse variation that our features can explain.

---
## Section 3: Data Preparation & Cleaning (Ch. 7)

For OLS regression on panel data (safehouse-months), we need to address:
1. Missing values in operational metrics (some months may have incomplete reporting).
2. Skewness in count variables (apply log1p transformation).
3. Outlier treatment for extreme values that could distort regression coefficients.
4. Proper encoding of the region categorical variable with reference category dropping.

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing Values:')
print(missing[missing > 0])

# Fill missing numeric features
for col in feature_cols:
    if col in df.columns and df[col].isnull().any():
        df = missing_fill(df, col, strategy='median')
        print(f'  Filled {col} with median')

# Fill missing target
if df['avg_education_progress'].isnull().any():
    df = df.dropna(subset=['avg_education_progress'])
    print(f'  Dropped rows with missing target. New shape: {df.shape}')

# Fill missing region
if df['region'].isnull().any():
    df = missing_fill(df, 'region', strategy='mode')

print(f'\nRemaining missing: {df.isnull().sum().sum()}')

In [ ]:
# Skewness correction (Ch. 7)
skew_candidates = ['process_recording_count', 'home_visitation_count', 'incident_count']
for col in skew_candidates:
    if col in df.columns and abs(df[col].skew()) > 1:
        original_skew = df[col].skew()
        df = skew_correct(df, col, method='log')
        print(f'{col}: skew {original_skew:.2f} -> {df[col].skew():.2f}')

In [ ]:
# Outlier treatment (Ch. 7)
for col in ['capacity_utilization', 'active_residents']:
    if col in df.columns:
        df = clean_outlier(df, col, method='iqr', factor=1.5)
        print(f'{col}: outliers capped via IQR')

In [ ]:
# Create modeling dataset with dummy variables (Ch. 7)
model_features = ['active_residents', 'process_recording_count', 'home_visitation_count',
                  'incident_count', 'partner_count', 'capacity_utilization', 'region']
target = 'avg_education_progress'

df_model = df[[target] + [f for f in model_features if f in df.columns]].copy()
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True, dtype=int)

print(f'Modeling dataset shape: {df_model.shape}')
print(f'Columns: {list(df_model.columns)}')

---
## Section 4: Bivariate Analysis (Ch. 8)

Chapter 8 bivariate analysis examines how each operational factor relates to education progress. This analysis directly answers leadership's key question: "What drives better education outcomes?"

- **N2N:** Correlations between numeric operational metrics and education progress.
- **N2C:** Whether education progress differs significantly by region.

In [ ]:
# N2N: Numeric features vs avg_education_progress (Ch. 8)
n2n_features = ['active_residents', 'process_recording_count', 'home_visitation_count',
                'incident_count', 'partner_count', 'capacity_utilization']

for col in n2n_features:
    if col in df.columns:
        result = n2n_analysis(df, col, target)
        print(f'{col} -> {target}: r={result["pearson_r"]:.3f}, p={result["p_value"]:.4f}\n')

**N2N Interpretation (Ch. 8):**

The Pearson correlations reveal the linear associations between operational inputs and education outcomes:

- **process_recording_count:** We expect a positive correlation -- more counseling sessions should support better education engagement as residents work through trauma that impedes learning.
- **home_visitation_count:** Positive correlation expected -- active family engagement provides stability that supports academic progress.
- **incident_count:** Negative correlation expected -- behavioral disruptions undermine the learning environment.
- **capacity_utilization:** Potentially negative -- overcrowded safehouses may have stretched resources and less individualized attention.
- **partner_count:** More partners may provide more specialized services (tutoring, mentoring) that boost education outcomes.

In [ ]:
# N2C: Education progress by region (Ch. 8)
if 'region' in df.columns:
    result = n2c_analysis(df, target, 'region')
    print(f'\navg_education_progress by region: F={result["f_statistic"]:.2f}, p={result["p_value"]:.4f}')

In [ ]:
# Correlation heatmap (Ch. 8)
corr_cols = [c for c in n2n_features + [target] if c in df.columns]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Operational Metrics Correlation Matrix (Ch. 8)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 5: Model Building (Ch. 9, 10)

### Primary Model: Statsmodels OLS with Standardized Coefficients (Ch. 9)

Because our features are measured on different scales (resident counts, session counts, ratios), raw coefficients are not directly comparable. We fit the OLS model on **standardized features** (z-scores) so that all coefficients are expressed in terms of standard deviation changes. This allows direct comparison: a coefficient of 0.3 means a one-standard-deviation increase in that feature is associated with a 0.3 standard deviation increase in education progress.

We also perform full regression diagnostics as required by Chapter 9: residual analysis, Q-Q plots, VIF, and Durbin-Watson.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

# Separate X and y
y = df_model[target]
X = df_model.drop(columns=[target])

# Standardize features for comparable coefficients
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_standardized = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

# Fit OLS
X_const = sm.add_constant(X_standardized)
ols_model = sm.OLS(y, X_const).fit()

print(ols_model.summary())

In [ ]:
# Coefficient table with standardized effects
coef_table = pd.DataFrame({
    'std_coefficient': ols_model.params,
    'std_error': ols_model.bse,
    't_statistic': ols_model.tvalues,
    'p_value': ols_model.pvalues,
    'ci_lower': ols_model.conf_int()[0],
    'ci_upper': ols_model.conf_int()[1]
})
coef_table['significant'] = coef_table['p_value'] < 0.05
coef_table['abs_effect'] = coef_table['std_coefficient'].abs()

print(f'Model R-squared: {ols_model.rsquared:.4f}')
print(f'Adjusted R-squared: {ols_model.rsquared_adj:.4f}')
print(f'F-statistic: {ols_model.fvalue:.2f} (p={ols_model.f_pvalue:.4e})')
print(f'\nStandardized Coefficients (Ch. 9):')
coef_table.drop(columns=['abs_effect']).sort_values('p_value')

**Standardized Coefficient Interpretation (Ch. 9):**

With standardized features, each coefficient represents the change in education progress (in original units) for a one-standard-deviation increase in that operational metric. This allows direct comparison of effect magnitudes:

The features with the **largest absolute standardized coefficients** are the most impactful operational levers. For example:
- If `process_recording_count` has the largest positive coefficient, investing in more counseling capacity would yield the greatest education improvement.
- If `incident_count` has a large negative coefficient, incident prevention programs are critical.
- If `capacity_utilization` has a negative coefficient, overcrowding directly harms education outcomes, justifying investment in additional capacity.

These coefficients form the basis of the **comparative dashboard recommendations** that leadership can use for resource allocation.

In [ ]:
# Visualize standardized coefficients
coef_plot = coef_table[coef_table.index != 'const'].sort_values('std_coefficient')

fig, ax = plt.subplots(figsize=(10, max(4, len(coef_plot) * 0.5)))
colors = ['#27AE60' if x > 0 else '#E74C3C' for x in coef_plot['std_coefficient']]
alpha_vals = [1.0 if sig else 0.4 for sig in coef_plot['significant']]

bars = ax.barh(coef_plot.index, coef_plot['std_coefficient'], color=colors)
for bar, alpha in zip(bars, alpha_vals):
    bar.set_alpha(alpha)

ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.set_xlabel('Standardized Coefficient (effect on avg_education_progress)')
ax.set_title('Standardized OLS Coefficients (Ch. 9)\nSolid = significant (p<0.05), Faded = not significant')
plt.tight_layout()
plt.show()

### 5.1 Regression Diagnostics (Ch. 9)

Full diagnostic suite for the OLS model. For panel data (repeated measures by safehouse), we pay special attention to the Durbin-Watson test, as temporal autocorrelation within safehouses could bias standard errors.

In [ ]:
# Regression diagnostics (Ch. 9)
residuals = ols_model.resid
fitted = ols_model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residual vs Fitted
axes[0, 0].scatter(fitted, residuals, alpha=0.4, color='#3498DB', s=20)
axes[0, 0].axhline(y=0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

# Q-Q Plot
stats.probplot(residuals, plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')

# Scale-Location
axes[1, 0].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.4, color='#E74C3C', s=20)
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('sqrt(|Residuals|)')
axes[1, 0].set_title('Scale-Location')

# Residual histogram
axes[1, 1].hist(residuals, bins=30, color='#91B191', edgecolor='white', density=True)
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()), 'r-', linewidth=2)
axes[1, 1].set_title('Residual Distribution')

plt.suptitle('OLS Regression Diagnostics (Ch. 9)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# VIF Analysis (Ch. 9)
vif_results = calculate_vif(X)
print('Variance Inflation Factors (Ch. 9):')
vif_results

In [ ]:
# Durbin-Watson test
dw = durbin_watson(residuals)
print(f'Durbin-Watson Statistic: {dw:.4f}')
if 1.5 < dw < 2.5:
    print('No significant autocorrelation detected.')
elif dw < 1.5:
    print('WARNING: Positive autocorrelation detected. This is expected with panel data (repeated safehouse measures).')
    print('Standard errors may be understated. Consider clustered standard errors or panel regression.')
else:
    print('Negative autocorrelation detected.')

**Diagnostics Interpretation (Ch. 9):**

- **Residual vs Fitted:** Random scatter supports the linearity assumption. Any curvature would suggest non-linear relationships requiring polynomial terms or feature transformations.
- **Q-Q Plot:** Deviations in the tails are acceptable given our sample size. The central portion following the line confirms approximate normality.
- **VIF:** Features with VIF > 5 indicate multicollinearity. If active_residents and capacity_utilization are collinear (utilization is derived from residents/capacity), we may need to drop one.
- **Durbin-Watson:** Values below 1.5 are expected with panel data and suggest that within-safehouse temporal correlation exists. For more rigorous inference, we would use clustered standard errors or a panel fixed-effects model.

### 5.2 Predictive Comparison: Ridge & Lasso Regression (Ch. 10)

Chapter 10 introduces regularized regression as a predictive alternative to OLS. Ridge (L2 penalty) shrinks coefficients toward zero, reducing overfitting. Lasso (L1 penalty) can set coefficients exactly to zero, performing automatic feature selection.

Comparing these with OLS provides robustness checks:
- If Ridge/Lasso perform similarly to OLS on test data, our explanatory model is stable.
- If Lasso zeros out certain features, those may be dispensable.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Models
regressors = {
    'OLS (Linear)': LinearRegression(),
    'Ridge': RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]),
    'Lasso': LassoCV(alphas=[0.01, 0.1, 1, 10], cv=5, random_state=42)
}

reg_results = {}
for name, model in regressors.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    reg_results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }
    print(f'{name:20s} | MAE: {reg_results[name]["MAE"]:.4f} | RMSE: {reg_results[name]["RMSE"]:.4f} | R2: {reg_results[name]["R2"]:.4f}')

reg_results_df = pd.DataFrame(reg_results).T
reg_results_df

In [ ]:
# Compare coefficients across models
coef_comparison = pd.DataFrame({
    'OLS': LinearRegression().fit(X_train_sc, y_train).coef_,
    'Ridge': regressors['Ridge'].coef_,
    'Lasso': regressors['Lasso'].coef_
}, index=X.columns)

print('Coefficient Comparison (standardized):')
coef_comparison

In [ ]:
# Visualize coefficient comparison
fig, ax = plt.subplots(figsize=(12, max(4, len(coef_comparison) * 0.5)))
x = np.arange(len(coef_comparison))
width = 0.25

ax.barh(x - width, coef_comparison['OLS'], width, label='OLS', color='#3498DB', alpha=0.8)
ax.barh(x, coef_comparison['Ridge'], width, label='Ridge', color='#E74C3C', alpha=0.8)
ax.barh(x + width, coef_comparison['Lasso'], width, label='Lasso', color='#27AE60', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(coef_comparison.index)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized Coefficient')
ax.set_title('Coefficient Comparison: OLS vs Ridge vs Lasso (Ch. 10)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Lasso feature selection: which features were zeroed out?
lasso_coefs = pd.Series(regressors['Lasso'].coef_, index=X.columns)
zeroed = lasso_coefs[lasso_coefs == 0]
kept = lasso_coefs[lasso_coefs != 0]

print(f'Lasso zeroed out {len(zeroed)} features: {list(zeroed.index)}')
print(f'Lasso kept {len(kept)} features: {list(kept.index)}')
print(f'Best alpha: {regressors["Lasso"].alpha_:.4f}')

**Predictive Comparison Interpretation (Ch. 10):**

The three models provide a robustness check for our explanatory findings:
- **OLS vs Ridge:** If the test-set metrics are similar, multicollinearity is not severely distorting our OLS estimates. If Ridge performs substantially better, some features are correlated enough to destabilize OLS.
- **Lasso Feature Selection:** Features zeroed out by Lasso are candidates for removal. If Lasso achieves comparable R2 with fewer features, the simpler model is preferred for interpretability.
- **Coefficient Stability:** Features whose coefficients remain consistent across all three methods are the most robust findings. Features with wildly different coefficients across methods may be unreliable.

---
## Section 6: Deployment - Comparative Dashboard Recommendations (Ch. 15)

The final deliverable is a set of actionable recommendations for the Foundation's leadership dashboard, derived from the model's standardized coefficients. Each recommendation links an operational lever to its estimated impact on education outcomes, enabling evidence-based resource allocation.

In [ ]:
# Build recommendation table
sig_features = coef_table[(coef_table.index != 'const') & (coef_table['significant'])].sort_values('abs_effect', ascending=False)

recommendations = []
for feat, row in sig_features.iterrows():
    direction = 'Increase' if row['std_coefficient'] > 0 else 'Decrease'
    impact = abs(row['std_coefficient'])
    recommendations.append({
        'Operational Factor': feat,
        'Effect Direction': direction,
        'Standardized Impact': round(impact, 4),
        'P-Value': round(row['p_value'], 4),
        'Recommendation': f'{direction} investment in {feat.replace("_", " ")} to improve education outcomes'
    })

rec_df = pd.DataFrame(recommendations)
print('Dashboard Recommendations (Ch. 15):')
rec_df

In [ ]:
# Save recommendations
rec_df.to_csv('models/safehouse_effectiveness_recommendations.csv', index=False)
print('Recommendations saved to models/safehouse_effectiveness_recommendations.csv')

# Save coefficient comparison for dashboard
coef_comparison.to_csv('models/safehouse_coefficient_comparison.csv')
print('Coefficient comparison saved to models/safehouse_coefficient_comparison.csv')

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Model comparison
reg_results_df[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], color=['#3498DB', '#E74C3C'])
axes[0].set_title('Predictive Model Comparison (Ch. 10)')
axes[0].set_ylabel('Error')
plt.sca(axes[0])
plt.xticks(rotation=45, ha='right')

# R2 comparison
reg_results_df['R2'].plot(kind='bar', ax=axes[1], color='#27AE60')
axes[1].set_title('R-squared Comparison')
axes[1].set_ylabel('R-squared')
axes[1].set_ylim(0, 1)
plt.sca(axes[1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

---
## Summary & Business Recommendations

### Key Findings

This pipeline identified the operational factors that most strongly influence education outcomes across the Lighthouse Foundation's safehouses. Using standardized OLS regression with full diagnostics, we quantified each factor's contribution and validated the findings against regularized models (Ridge, Lasso).

### Strategic Recommendations for Leadership

1. **Resource Allocation Formula:** Use the standardized coefficients as weights in a resource allocation model. Safehouses with lower scores on high-impact factors should receive priority investment.

2. **Capacity Management:** If capacity utilization shows a negative effect, the Foundation should establish a maximum utilization threshold (e.g., 85%) beyond which new admissions are diverted to other facilities. This protects education quality for current residents.

3. **Counseling Investment:** If process recording count has a strong positive effect, increasing the social worker-to-resident ratio at underperforming safehouses should be a priority. Each additional standard deviation of counseling sessions corresponds to a measurable increase in education progress.

4. **Incident Prevention Programs:** If incident count has a negative effect, the Foundation should invest in proactive behavioral support programs. The model quantifies the education cost of incidents, providing ROI justification for prevention investments.

5. **Partner Engagement:** If partner count shows a positive effect, the Foundation should actively recruit additional partners for safehouses with fewer partnerships, particularly in education and tutoring program areas.

6. **Regional Best Practices:** Significant region coefficients suggest that some regions have structural advantages or disadvantages. The Foundation should facilitate knowledge transfer from high-performing regions to lower-performing ones.

### Model Limitations

- **Panel structure:** The repeated safehouse-month observations violate the independence assumption of OLS. A panel regression with safehouse fixed effects would be more rigorous.
- **Omitted variables:** Factors like staff quality, curriculum design, and individual resident characteristics are not captured at the safehouse-month level.
- **Causality:** Regression identifies associations, not causation. Process recording count may correlate with education progress because well-managed safehouses invest in both, not because counseling directly causes better education outcomes.